# Generation Pipeline — End-to-End Test

Tests the full `RAGPipeline` stack: hybrid retrieval → cross-encoder rerank → Ollama LLM → structured answer.

**Three test questions (designed by difficulty):**
- **Q1** — well-covered: *RAGAS faithfulness metric* — should retrieve directly from `ragas.pdf`
- **Q2** — tangential: *energy cost of training large neural networks* — ML papers may touch this but don't focus on it
- **Q3** — out of scope: *best programming language for REST APIs* — completely absent from the corpus

**Cross-collection comparison:** Q1 is run across all three collections (`rag_fixed`, `rag_recursive`, `rag_semantic`) to show how chunking strategy affects the retrieved context and the generated answer.

In [1]:
import sys, time, textwrap
sys.path.insert(0, '..')

from configs.settings import AppConfig
from src.embedding.embedder import Embedder
from src.vectorstore.store import VectorStore
from src.generation import RAGPipeline

cfg = AppConfig()
print(f'Ollama endpoint : {cfg.ollama_base_url}')
print(f'Model           : {cfg.ollama_model}')
print(f'API key set     : {bool(cfg.ollama_api_key and cfg.ollama_api_key != "your_key_here")}')
print(f'Retrieval limit : {cfg.retrieval_limit}  →  rerank top {cfg.rerank_top_n}')
print(f'Max context     : {cfg.max_context_chars} chars')

/Users/harsh/drive/workspace/ai_projects/doc-rag-engine/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ollama endpoint : http://localhost:11434
Model           : llama3.1:8b
API key set     : False
Retrieval limit : 20  →  rerank top 5
Max context     : 12000 chars


In [2]:
# Shared components — construct once, reuse across all pipeline instances
print('Loading embedder (MiniLM)...')
embedder = Embedder()

print('Connecting to Qdrant...')
store = VectorStore()

print('Done.')

Loading embedder (MiniLM)...


Connecting to Qdrant...
Done.


In [3]:
# Build the pipeline on rag_recursive (12 389 pts).
# BM25 index build + FlashRank model load happen here — slow once.
print('Building RAGPipeline (BM25 index + FlashRank load)...')
t0 = time.time()
pipeline = RAGPipeline(embedder, store, cfg.collection_recursive, cfg)
print(f'Ready in {time.time() - t0:.1f}s')

Building RAGPipeline (BM25 index + FlashRank load)...


INFO:src.retrieval.reranker:Rerank model ready.


Ready in 0.5s


## Helper

In [4]:
def show_result(result: dict, elapsed: float) -> None:
    """Pretty-print a RAGPipeline.invoke() result dict."""
    print(f"Query      : {result['query']}")
    print(f"Collection : {result['collection']}  |  method: {result['retrieval_method']}")
    print(f"Time       : {elapsed:.2f}s")
    if result.get('error'):
        print(f"\n*** ERROR: {result['error']}")
    print(f"\nAnswer:\n{textwrap.fill(result['answer'], width=90)}")
    print(f"\nSource chunks ({len(result['source_chunks'])}):")
    for i, c in enumerate(result['source_chunks'], 1):
        fname  = c.metadata.get('filename', 'unknown')
        cidx   = c.metadata.get('chunk_index', '?')
        strat  = c.metadata.get('chunk_strategy', '?')
        score  = f"{c.score:.4f}"
        snippet = c.chunk_text[:120].replace('\n', ' ')
        print(f"  [{i}] {fname} | chunk {cidx} | strategy={strat} | score={score}")
        print(f"      {snippet}...")
    print()

## Q1 — Well-covered: RAGAS faithfulness metric

**Expected:** answer grounded in `ragas.pdf`, defines faithfulness as the fraction of claims in the answer that are supported by the retrieved context. Should cite source [1] or [2].

In [5]:
Q1 = "What is the RAGAS faithfulness metric and how is it calculated?"

t0 = time.time()
r1 = pipeline.invoke(Q1)
elapsed = time.time() - t0

show_result(r1, elapsed)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.86it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.85it/s]


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_recursive/points/query "HTTP/1.1 200 OK"


ERROR:src.generation.pipeline:LLM call failed  question='What is the RAGAS faithfulness metric and how is it calculated?'  error=[Errno 61] Connection refused


INFO:src.generation.pipeline:invoke complete  method=hybrid_rrf  chunks=5  error=[Errno 61] Connection refused


Query      : What is the RAGAS faithfulness metric and how is it calculated?
Collection : rag_recursive  |  method: hybrid_rrf
Time       : 0.42s

*** ERROR: [Errno 61] Connection refused

Answer:
Generation failed: the language model did not return a response.

Source chunks (5):
  [1] ragas.pdf | chunk 51 | strategy=recursive | score=0.9732
      To put the results in context, we compare our proposed metrics (shown as Ragas in Table 1) with two baseline methods. Fo...
  [2] ragas.pdf | chunk 2 | strategy=recursive | score=0.9497
      . Evaluating RAG architectures is, however, challenging because there are several dimensions to consider: the ability of...
  [3] ragas.pdf | chunk 52 | strategy=recursive | score=0.8982
      Faithfulness measures the information consistency of the answer against the given context. Any claims that are made in t...
  [4] ragas.pdf | chunk 26 | strategy=recursive | score=0.8951
      . First, Faithfulness refers to the idea that the answer should be grou

**Observations — Q1 (RAGAS faithfulness):**

Retrieval worked correctly. Ollama is not running locally (`Connection refused`), so LLM generation failed — answers below are based on what *would* be sent to the model.

**Retrieval signal:**
All 5 retrieved chunks came from `ragas.pdf` — no cross-document noise. Reranker scores were consistently high:

| Rank | File | Chunk | Score | Snippet |
|---|---|---|---|---|
| 1 | ragas.pdf | 51 | **0.9732** | "compare our proposed metrics (shown as Ragas in Table 1)..." |
| 2 | ragas.pdf | 2 | 0.9497 | "Evaluating RAG architectures is challenging..." |
| 3 | ragas.pdf | 52 | 0.8982 | "Faithfulness measures information consistency of the answer against context..." |
| 4 | ragas.pdf | 26 | 0.8951 | "Faithfulness refers to the idea that the answer should be grounded..." |
| 5 | ragas.pdf | 60 | 0.8241 | "predictions from Ragas are closely aligned with human predictions..." |

Chunks 52 and 26 both contain the faithfulness *definition* — exactly what the question asks for. If Ollama were running, the context passed to the LLM would contain a complete, citable answer.

**Latency breakdown:** retrieval + rerank = 0.35 s. LLM generation would add 5–15 s depending on the model.

## Q2 — Tangential: Energy cost of training large neural networks

**Expected:** ML papers (attention, BERT, etc.) may mention compute requirements or parameter counts but are unlikely to discuss energy consumption directly. The answer should either be partial (citing compute cost numbers without energy framing) or use the no-answer response.

In [6]:
Q2 = "What are the energy or computational costs of training large neural networks like transformers?"

t0 = time.time()
r2 = pipeline.invoke(Q2)
elapsed = time.time() - t0

show_result(r2, elapsed)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 28.82it/s]


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_recursive/points/query "HTTP/1.1 200 OK"


ERROR:src.generation.pipeline:LLM call failed  question='What are the energy or computational costs of training large neural networks lik'  error=[Errno 61] Connection refused


INFO:src.generation.pipeline:invoke complete  method=hybrid_rrf  chunks=5  error=[Errno 61] Connection refused


Query      : What are the energy or computational costs of training large neural networks like transformers?
Collection : rag_recursive  |  method: hybrid_rrf
Time       : 0.25s

*** ERROR: [Errno 61] Connection refused

Answer:
Generation failed: the language model did not return a response.

Source chunks (5):
  [1] long_context_llm.pdf | chunk 10 | strategy=recursive | score=0.0058
      Handling these use-cases requires language models to successfully operate over long sequences. Existing language models ...
  [2] long_context_llm.pdf | chunk 106 | strategy=recursive | score=0.0016
      There is much prior work in designing performant language models with cheaper scaling than Transformers in the context l...
  [3] text_embed.pdf | chunk 125 | strategy=recursive | score=0.0007
      - [23] Bo Han, Quanming Yao, Xingrui Yu, Gang Niu, Miao Xu, Weihua Hu, Ivor W. Tsang, and Masashi Sugiyama. Co-teaching:...
  [4] attention_need.pdf | chunk 73 | strategy=recursive | score=0.0006
      

**Observations — Q2 (Energy cost of training, tangential):**

**Retrieval signal is weak — exactly as designed:**

| Rank | File | Chunk | Score | Snippet |
|---|---|---|---|---|
| 1 | long_context_llm.pdf | 10 | **0.0058** | "requires language models to successfully operate over long sequences..." |
| 2 | long_context_llm.pdf | 106 | 0.0016 | "prior work in designing performant language models with cheaper scaling..." |
| 3 | text_embed.pdf | 125 | 0.0007 | (references section) |
| 4 | attention_need.pdf | 73 | 0.0006 | WMT 2014 translation BLEU scores |
| 5 | attention_need.pdf | 4 | 0.0004 | WMT 2014 French translation results |

Top score of **0.0058 vs 0.9732 for Q1** — two orders of magnitude lower. The reranker found no passage that directly answers the question. Content about "cheaper scaling" and "performant language models" is adjacent but not about energy cost.

**What a working LLM should do:** the grounding instruction says to reply "The context does not contain enough information..." when the context doesn't answer the question. With scores this low, a well-prompted model should refuse rather than extrapolate.

**What a poorly prompted model would do:** hallucinate numbers from its training data — exactly the failure mode the grounding instruction is designed to prevent.

## Q3 — Out of scope: REST API programming language

**Expected:** corpus contains ML papers, Gutenberg classics, and economic reports — nothing about web development. Pipeline should retrieve weakly-matching chunks, and the LLM should respond with the no-answer phrase exactly: *"The context does not contain enough information to answer this question."*

In [7]:
Q3 = "What is the best programming language for building REST APIs?"

t0 = time.time()
r3 = pipeline.invoke(Q3)
elapsed = time.time() - t0

show_result(r3, elapsed)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 26.44it/s]


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_recursive/points/query "HTTP/1.1 200 OK"


ERROR:src.generation.pipeline:LLM call failed  question='What is the best programming language for building REST APIs?'  error=[Errno 61] Connection refused


INFO:src.generation.pipeline:invoke complete  method=hybrid_rrf  chunks=5  error=[Errno 61] Connection refused


Query      : What is the best programming language for building REST APIs?
Collection : rag_recursive  |  method: hybrid_rrf
Time       : 0.26s

*** ERROR: [Errno 61] Connection refused

Answer:
Generation failed: the language model did not return a response.

Source chunks (5):
  [1] llm_eval.pdf | chunk 305 | strategy=recursive | score=0.0008
      dialogues, encompassing a total of 568 API calls. Furthermore, the ToolBench project [191] aims to empower the developme...
  [2] rag_llm.pdf | chunk 360 | strategy=recursive | score=0.0005
      - [160] Y. Hoshi, D. Miyashita, Y. Ng, K. Tatsuno, Y. Morioka, O. Torii, and J. Deguchi, 'Ralle: A framework for develop...
  [3] llm_eval.pdf | chunk 217 | strategy=recursive | score=0.0004
      . Toolformer employs a training approach to determine the optimal usage of specific APIs and integrates the obtained res...
  [4] rag_llm.pdf | chunk 262 | strategy=recursive | score=0.0003
      The development of the RAG ecosystem is greatly impacted b

**Observations — Q3 (REST APIs, out of scope):**

**Retrieval shows semantic drift via keyword collision:**

| Rank | File | Chunk | Score | Snippet |
|---|---|---|---|---|
| 1 | llm_eval.pdf | 305 | 0.0008 | "568 API calls... ToolBench project aims to empower..." |
| 2 | rag_llm.pdf | 360 | 0.0005 | (references section) |
| 3 | llm_eval.pdf | 217 | 0.0004 | "Toolformer... determines optimal usage of specific APIs..." |
| 4 | rag_llm.pdf | 262 | 0.0003 | "Key tools like LangChain..." |
| 5 | llm_eval.pdf | 41 | 0.0003 | "Transformers have revolutionized the field..." |

The word "API" appears in ML evaluation papers (model APIs, ToolBench, Toolformer) — a completely different domain from REST web APIs. The retriever correctly signals near-zero confidence (scores 0.0003–0.0008). This is the system working: it found the keyword but the reranker knew the semantic match was wrong.

**Score comparison across all three queries:**
- Q1 (in scope): top score **0.9732**
- Q2 (tangential): top score **0.0058** (167× lower)
- Q3 (out of scope): top score **0.0008** (1216× lower)

This score spread is a natural threshold signal for the evaluation dashboard: scores below ~0.01 strongly suggest the query is out of distribution.

## Cross-collection comparison — Q1 across all three strategies

Same question, three collections. **Fixed-size** may split mid-sentence and lose context. **Recursive** respects paragraph breaks. **Semantic** creates larger, coherent topic chunks. Expect qualitatively different context windows and potentially different answers.

In [8]:
# Build pipelines for fixed and semantic; recursive is already built above.
# BM25 indices build per-collection so these take ~10s each.
print('Building pipeline for rag_fixed...')
t0 = time.time()
pipeline_fixed = RAGPipeline(embedder, store, cfg.collection_fixed, cfg)
print(f'  ready in {time.time()-t0:.1f}s')

print('Building pipeline for rag_semantic...')
t0 = time.time()
pipeline_semantic = RAGPipeline(embedder, store, cfg.collection_semantic, cfg)
print(f'  ready in {time.time()-t0:.1f}s')

INFO:src.generation.pipeline:Building RAGPipeline  collection=rag_fixed  model=llama3.1:8b  url=http://localhost:11434


INFO:src.retrieval.sparse:Building BM25 index from rag_fixed...


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_fixed/points/scroll "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_fixed/points/scroll "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_fixed/points/scroll "HTTP/1.1 200 OK"


...

INFO:src.vectorstore.store:Scrolled 16386 points from rag_semantic


INFO:src.retrieval.sparse:BM25 index ready — 16386 documents


INFO:src.retrieval.reranker:Loading rerank model ms-marco-MiniLM-L-12-v2...


INFO:src.retrieval.reranker:Rerank model ready.


  ready in 0.5s


In [9]:
print('=' * 70)
print('FIXED-SIZE  (rag_fixed, 10 439 pts)')
print('=' * 70)
t0 = time.time()
rf = pipeline_fixed.invoke(Q1)
show_result(rf, time.time() - t0)

print('=' * 70)
print('RECURSIVE   (rag_recursive, 12 389 pts)')
print('=' * 70)
t0 = time.time()
rr = pipeline.invoke(Q1)   # already built above
show_result(rr, time.time() - t0)

print('=' * 70)
print('SEMANTIC    (rag_semantic, 16 386 pts)')
print('=' * 70)
t0 = time.time()
rs = pipeline_semantic.invoke(Q1)
show_result(rs, time.time() - t0)

FIXED-SIZE  (rag_fixed, 10 439 pts)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 116.90it/s]


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_fixed/points/query "HTTP/1.1 200 OK"


ERROR:src.generation.pipeline:LLM call failed  question='What is the RAGAS faithfulness metric and how is it calculated?'  error=[Errno 61] Connection refused


...

Batches: 100%|██████████| 1/1 [00:00<00:00, 135.76it/s]


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_semantic/points/query "HTTP/1.1 200 OK"


ERROR:src.generation.pipeline:LLM call failed  question='What is the RAGAS faithfulness metric and how is it calculated?'  error=[Errno 61] Connection refused


INFO:src.generation.pipeline:invoke complete  method=hybrid_rrf  chunks=5  error=[Errno 61] Connection refused


Query      : What is the RAGAS faithfulness metric and how is it calculated?
Collection : rag_semantic  |  method: hybrid_rrf
Time       : 0.21s

*** ERROR: [Errno 61] Connection refused

Answer:
Generation failed: the language model did not return a response.

Source chunks (5):
  [1] ragas.pdf | chunk 57 | strategy=semantic | score=0.9822
      [question]  answer 1 :  [answer 1]  answer 2 : [answer 2]  The results in Table 1 show that our proposed metrics are muc...
  [2] ragas.pdf | chunk 2 | strategy=semantic | score=0.9437
      . Evaluating RAG architectures is, however, challenging because there are several dimensions to consider: the ability of...
  [3] ragas.pdf | chunk 26 | strategy=semantic | score=0.9242
      . First, Faithfulness refers to the idea that the answer should be grounded in the given context. This is important to a...
  [4] ragas.pdf | chunk 54 | strategy=semantic | score=0.9050
      Faithfulness measures the information consistency of the answer against the 

**Cross-collection observations — Q1 (RAGAS faithfulness):**

| Collection | Top-1 score | Top-1 chunk | 5th chunk score | Cross-doc noise? |
|---|---|---|---|---|
| rag_fixed | 0.9447 | ragas.pdf chunk 2 | 0.2305 | **Yes** — chunk 4 is from rag_llm.pdf |
| rag_recursive | 0.9732 | ragas.pdf chunk 51 | 0.8241 | No — all 5 from ragas.pdf |
| rag_semantic | **0.9822** | ragas.pdf chunk 57 | 0.7862 | No — all 5 from ragas.pdf |

**Three findings:**

1. **Fixed-size contamination.** `rag_fixed` chunk 4 is from `rag_llm.pdf` (score 0.4087), not `ragas.pdf`. Hard character boundaries split passages mid-thought, so a chunk that opens with "evaluation objectives" lands in the wrong semantic neighbourhood. The fixed collection would give the LLM a diluted context.

2. **Fixed-size truncation artifact.** `rag_fixed` chunk 2 starts with "he retrieval system..." — the word "the" was split across a chunk boundary. Exactly the kind of mid-word cut that fixed chunking is prone to.

3. **Semantic scored highest but retrieved results content, not definition.** `rag_semantic` top-1 (chunk 57, score 0.9822) contains comparison table results ("The results in Table 1 show that our proposed metrics are much better..."), while `rag_recursive` top-1 (chunk 51) + chunks 3–4 contain the *definition* of faithfulness. For a "what is X" question, definitions are more useful than results. **High reranker score ≠ most useful for generation.** This is the key finding to carry into the evaluation module.

**Implication for the dashboard:** show reranker scores alongside chunk previews so users can inspect what each collection actually retrieved, not just the final answer.

## Streaming demo

Confirms that `stream()` yields tokens progressively. This is the mode the Streamlit dashboard will use.

In [10]:
print(f'Streaming answer for: "{Q1}"\n')
print('-' * 60)
t0 = time.time()
for token in pipeline.stream(Q1):
    print(token, end='', flush=True)
print(f'\n{"-" * 60}')
print(f'Total time: {time.time() - t0:.2f}s')

Streaming answer for: "What is the RAGAS faithfulness metric and how is it calculated?"

------------------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 182.45it/s]


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/rag_recursive/points/query "HTTP/1.1 200 OK"


INFO:src.generation.pipeline:stream started  method=hybrid_rrf  chunks=5


ERROR:src.generation.pipeline:LLM stream failed  question='What is the RAGAS faithfulness metric and how is it calculated?'  error=[Errno 61] Connection refused




[Generation failed: [Errno 61] Connection refused]


------------------------------------------------------------
Total time: 0.21s


## Error handling verification

Confirms the three edge-case guards return the correct structured shape without raising.

In [11]:
from unittest.mock import MagicMock

# --- guard 1: empty retrieval ---
original_retrieve = pipeline._retrieve
pipeline._retrieve = lambda *a, **kw: []
res = pipeline.invoke('anything')
pipeline._retrieve = original_retrieve
assert res['source_chunks'] == []
assert res['error'] is None
print('Guard 1 (empty retrieval):', res['answer'][:80])

# --- guard 2: LLM timeout ---
# RunnableSequence does not allow attribute-setting on instances, so we replace
# the entire _llm_chain with a MagicMock rather than patching its .invoke method.
from src.models import RetrievalResult
fake_chunk = RetrievalResult(
    chunk_text='Some context text.',
    score=0.9,
    metadata={'filename': 'test.pdf', 'chunk_index': 0},
    retrieval_method='reranked',
)
original_chain = pipeline._llm_chain
pipeline._retrieve = lambda *a, **kw: [fake_chunk]
pipeline._llm_chain = MagicMock()
pipeline._llm_chain.invoke = MagicMock(side_effect=TimeoutError('timed out'))
res = pipeline.invoke('anything')
pipeline._llm_chain = original_chain
pipeline._retrieve = original_retrieve
assert res['error'] == 'timed out'
assert len(res['source_chunks']) == 1
print('Guard 2 (LLM timeout):    ', res['answer'][:80])

# --- guard 3: context truncation ---
big_chunks = [
    RetrievalResult(
        chunk_text='x' * 5000,
        score=0.9 - i * 0.01,
        metadata={'filename': f'doc{i}.pdf', 'chunk_index': i},
        retrieval_method='reranked',
    )
    for i in range(4)
]
trimmed = pipeline._truncate_to_limit(big_chunks, 'test')
assert sum(len(r.chunk_text) for r in trimmed) <= cfg.max_context_chars
print(f'Guard 3 (truncation):     {len(big_chunks)} chunks → {len(trimmed)} chunks (limit {cfg.max_context_chars} chars)')

print('\nAll guards verified.')

ERROR:src.generation.pipeline:LLM call failed  question='anything'  error=timed out


INFO:src.generation.pipeline:invoke complete  method=hybrid_rrf  chunks=1  error=timed out


Guard 1 (empty retrieval): No relevant context was found in the documents for this query. Try rephrasing or
Guard 2 (LLM timeout):     Generation failed: the language model did not return a response.
Guard 3 (truncation):     4 chunks → 2 chunks (limit 12000 chars)

All guards verified.
